<a href="https://colab.research.google.com/github/rohithsai199/python-basics-day-1/blob/main/fast_API_day_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pydantic

In [2]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title="Student Management API")

# Temporary in-memory database
students_db = [
    {"id": 1, "name": "Alice", "age": 20, "grade": "A"},
    {"id": 2, "name": "Bob", "age": 22, "grade": "B"}
]

# Pydantic model for request validation
class Student(BaseModel):
    id: int
    name: str
    age: int
    grade: str

class StudentUpdate(BaseModel):
    name: Optional[str] = None
    age: Optional[int] = None
    grade: Optional[str] = None

# 1. GET: Fetch all students
@app.get("/students", response_model=List[dict])
def get_students():
    return students_db

# 2. GET: Fetch a single student by ID
@app.get("/students/{student_id}")
def get_student(student_id: int):
    for student in students_db:
        if student["id"] == student_id:
            return student
    raise HTTPException(status_code=404, detail="Student not found")

# 3. POST: Add a new student
@app.post("/students", status_code=201)
def create_student(student: Student):
    for existing in students_db:
        if existing["id"] == student.id:
            raise HTTPException(status_code=400, detail="Student ID already exists")

    new_student = student.dict()
    students_db.append(new_student)
    return {"message": "Student created successfully", "student": new_student}

# 4. PUT: Update an existing student
@app.put("/students/{student_id}")
def update_student(student_id: int, updated_data: StudentUpdate):
    for student in students_db:
        if student["id"] == student_id:
            if updated_data.name is not None:
                student["name"] = updated_data.name
            if updated_data.age is not None:
                student["age"] = updated_data.age
            if updated_data.grade is not None:
                student["grade"] = updated_data.grade
            return {"message": "Student updated successfully", "student": student}

    raise HTTPException(status_code=404, detail="Student not found")

# 5. DELETE: Remove a student
@app.delete("/students/{student_id}")
def delete_student(student_id: int):
    for index, student in enumerate(students_db):
        if student["id"] == student_id:
            students_db.pop(index)
            return {"message": "Student deleted successfully"}

    raise HTTPException(status_code=404, detail="Student not found")

In [3]:
import uvicorn
import asyncio

# Step 1: Print your IP address (required as password for localtunnel)
!curl ipv4.icanhazip.com

# Step 2: Start localtunnel on port 8000 in background
get_ipython().system_raw('npx localtunnel --port 8000 &')

# Step 3: Run the FastAPI server inside Colab
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

34.74.179.50


INFO:     Started server process [450]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [450]


In [4]:
!curl http://127.0.0.1:4040/api/tunnels 2>/dev/null | grep -o 'https://[^"]*loca.lt' || echo "Check localtunnel output"

Check localtunnel output


In [5]:
!npx localtunnel --port 8000

⠙⠹⠸⠼⠴your url is: https://true-walls-guess.loca.lt
^C


In [6]:
!pkill -f uvicorn
!pkill -f localtunnel

In [ ]:
import uvicorn
import asyncio
from subprocess import Popen, PIPE

# 1. Start localtunnel background process
tunnel = Popen(["npx", "localtunnel", "--port", "8000"], stdout=PIPE, stderr=PIPE)

# 2. Print your IP Address (Tunnel Password)
!curl ipv4.icanhazip.com

# 3. Read and print the localtunnel URL automatically
import time
time.sleep(3) # Give localtunnel 3 seconds to generate URL
output = tunnel.stdout.readline().decode('utf-8')
print("Your Public URL:", output.strip())
print("Your Swagger UI link:", output.strip() + "/docs")

# 4. Start FastAPI server
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

34.74.179.50
Your Public URL: your url is: https://good-onions-learn.loca.lt
Your Swagger UI link: your url is: https://good-onions-learn.loca.lt/docs


INFO:     Started server process [450]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     106.222.230.224:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     106.222.230.224:0 - "GET /openapi.json HTTP/1.1" 200 OK
